In [1]:
calls = 0
def fib(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(30), "computed in", calls, "calls")

832040 computed in 2692537 calls


In [2]:
from functools import lru_cache


@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(30))
print(fib.cache_info())

832040
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


In [3]:
import time

# Three functions with deliberately different complexity classes.
def constant_op(data):          # O(1): touch one element
    return data[0]

def linear_op(data):            # O(n): sum every element
    total = 0
    for x in data:
        total += x
    return total

def quadratic_op(data):         # O(n^2): compare every pair
    count = 0
    for a in data:
        for b in data:
            if a == b:
                count += 1
    return count

def time_it(func, n):
    data = list(range(n))
    start = time.perf_counter()
    func(data)
    return time.perf_counter() - start

sizes = [200, 400, 800, 1600, 3200]
for label, fn in [("O(1)", constant_op), ("O(n)", linear_op), ("O(n^2)", quadratic_op)]:
    row = [f"{time_it(fn, n)*1000:8.3f}" for n in sizes]
    print(f"{label:7}", "  ".join(row), "ms")
print("\nColumns = n:", sizes)

O(1)       0.001     0.003     0.000     0.000     0.000 ms
O(n)       0.008     0.018     0.037     0.074     0.109 ms
O(n^2)     0.706     2.880    10.045    37.080   152.399 ms

Columns = n: [200, 400, 800, 1600, 3200]


##### Time Complexities in increasing order of magnitude
* O(1)
* O(log n)
* O(n)
* O(n log n)
* O(n^2)
* O(n^3)
* O(2^n)
* O(k^n)
* O(n!)



## Arrays

* Homogeneous data type
* Contiguous
* Fixed size

**Array costs:**

| Operation | Cost | Why |
|---|---|---|
| index `lst[i]` | O(1) | address arithmetic |
| append | O(1)* | amortised; occasional resize |
| insert at front | **O(n)** | shift everything right |
| delete at front | **O(n)** | shift everything left |
| search by value | O(n) | scan |

In [4]:
sample = [10, 20, 30]
for i, value in enumerate(sample):
    print(f"index {i}: value={value}  id(value)={id(value)}")

index 0: value=10  id(value)=140727643763784
index 1: value=20  id(value)=140727643764104
index 2: value=30  id(value)=140727643764424


## Linked Lists

In [5]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class LinkedList:
    def __init__(self):
        self.head = None          # entry point to the chain

    def prepend(self, value):     # add at FRONT — O(1), just rewire head
        node = Node(value)
        node.next = self.head
        self.head = node

    def append(self, value):      # add at END — O(n), must walk to the tail
        node = Node(value)
        if self.head is None:
            self.head = node
            return
        cur = self.head
        while cur.next is not None:
            cur = cur.next
        cur.next = node

    def display(self):            # walk the chain, collect values
        out, cur = [], self.head
        while cur is not None:
            out.append(cur.value)
            cur = cur.next
        return out
    
#-----------------------------------------------------------------

chain = LinkedList()
for amount in [1500, 320, 9800]:
    chain.append(amount)
chain.prepend(75)                 # newest-first arrival at the front

print("chain:", chain.display())  # [75, 1500, 320, 9800]

chain: [75, 1500, 320, 9800]


In [6]:
N = 60000

# Array (list): insert at front N times -> each insert shifts everything -> O(n) each
arr = []
start = time.perf_counter()
for i in range(N):
    arr.insert(0, i)              # the expensive line: O(n) shift
arr_time = time.perf_counter() - start

# Linked list: prepend N times -> each is O(1) pointer rewiring
ll = LinkedList()
start = time.perf_counter()
for i in range(N):
    ll.prepend(i)                 # the cheap line: O(1)
ll_time = time.perf_counter() - start

print(f"list.insert(0, x)  x{N}: {arr_time*1000:8.1f} ms   (O(n) each)")
print(f"linkedlist.prepend x{N}: {ll_time*1000:8.1f} ms   (O(1) each)")
print(f"\nlinked list was ~{arr_time/ll_time:.0f}x faster for front insertion")

list.insert(0, x)  x60000:   1630.1 ms   (O(n) each)
linkedlist.prepend x60000:    344.0 ms   (O(1) each)

linked list was ~5x faster for front insertion


| | Array (`list`) | Linked List |
|---|---|---|
| Index access `[i]` | **O(1)** | O(n) — must walk |
| Insert/delete at front | O(n) | **O(1)** |
| Insert/delete at end | O(1)* | O(n) without a tail pointer |
| Memory | compact, contiguous | extra pointer per node |
| Cache behaviour | excellent (locality) | poor (scattered) |


## Stacks (LIFO) & Queues (FIFO)

In [8]:
class Stack:
    """A simple LIFO stack backed by a Python list (top = end of list)."""

    def __init__(self):
        self._items = []          # the end of the list is the "top"

    def push(self, item):
        self._items.append(item)  # O(1) amortized

    def pop(self):
        if self.is_empty():
            raise IndexError("pop from an empty stack")
        return self._items.pop()  # removes and returns the top

    def peek(self):
        if self.is_empty():
            raise IndexError("peek into an empty stack")
        return self._items[-1]    # look at the top without removing

    def is_empty(self):
        return len(self._items) == 0

    def size(self):
        return len(self._items)

    def data(self):
        return len(self._items)

    def __repr__(self):
        # bottom -> top, so the rightmost item is the top
        return f"Stack(bottom -> top: {self._items})"
    
#-------------------------------------------------------------------------------------

s = Stack()
print("Empty at start? ", s.is_empty())   # True

for x in (10, 20, 30):
    s.push(x)
    print(f"pushed {x:>2} -> {s}")

print("Top (peek):     ", s.peek())        # 30, stack unchanged
print("Size:           ", s.size())        # 3

print("popped:         ", s.pop())         # 30
print("popped:         ", s.pop())         # 20
print("After two pops: ", s.data())               # bottom -> top: [10]

print("Empty now?      ", s.is_empty())    # False

Empty at start?  True
pushed 10 -> Stack(bottom -> top: [10])
pushed 20 -> Stack(bottom -> top: [10, 20])
pushed 30 -> Stack(bottom -> top: [10, 20, 30])
Top (peek):      30
Size:            3
popped:          30
popped:          20
After two pops:  1
Empty now?       False


In [9]:
# A rollback log: the last action applied is the first to be undone.
rollback = []
for action in ["debit-500", "credit-200", "fee-15"]:
    rollback.append(action)       # push
    print("pushed:", action)

print("\nundo order (LIFO):")
while rollback:
    print("  undo ->", rollback.pop())   # pop from the top

pushed: debit-500
pushed: credit-200
pushed: fee-15

undo order (LIFO):
  undo -> fee-15
  undo -> credit-200
  undo -> debit-500


In [ ]:
# Expression Conversion
# Infix to Posfix

def infix_to_postfix(tokens):
    prec = {"+": 1, "-": 1, "*": 2, "/": 2, "^": 3}
    output, ops = [], []          # output list + operator stack
    for tok in tokens:
        if tok in prec:                                   # an operator
            while ops and ops[-1] != "(" and prec.get(ops[-1], 0) >= prec[tok]:
                output.append(ops.pop())                  # pop higher/equal precedence
            ops.append(tok)
        elif tok == "(":
            ops.append(tok)
        elif tok == ")":
            while ops and ops[-1] != "(":
                output.append(ops.pop())
            ops.pop()                                     # discard the "("
        else:                                             # an operand
            output.append(tok)
    while ops:
        output.append(ops.pop())
    return output

# "principal + rate * years"  ->  operands and operators as tokens
expr = ["principal", "+", "rate", "*", "years"]
print("infix  :", " ".join(expr))
print("postfix:", " ".join(infix_to_postfix(expr)))

expr2 = ["(", "principal", "+", "rate", ")", "*", "years"]
print("\ninfix  :", " ".join(expr2))
print("postfix:", " ".join(infix_to_postfix(expr2)))

### QUEUES (FIFO)

In [ ]:
# Using List - NOT a good option really.
# High cost of list.pop(0)
# 
# Loan applications arrive; we process them in arrival order.
pipeline = []
for app_id in ["APP-1001", "APP-1002", "APP-1003"]:
    pipeline.append(app_id)       # enqueue at the back — O(1)
    print("received:", app_id)

print("\nprocessing order (FIFO):")
while pipeline:
    print("  processing ->", pipeline.pop(0))   # dequeue from the front — O(n)

received: APP-1001
received: APP-1002
received: APP-1003

processing order (FIFO):
  processing -> APP-1001
  processing -> APP-1002
  processing -> APP-1003


In [12]:
from collections import deque

# Loan applications arrive; we process them in arrival order.
pipeline = deque()
for app_id in ["APP-1001", "APP-1002", "APP-1003"]:
    pipeline.append(app_id)       # enqueue at the back — O(1)
    print("received:", app_id)

print("\nprocessing order (FIFO):")
while pipeline:
    print("  processing ->", pipeline.popleft())   # dequeue from the front — O(1)

received: APP-1001
received: APP-1002
received: APP-1003

processing order (FIFO):
  processing -> APP-1001
  processing -> APP-1002
  processing -> APP-1003


In [1]:
range(10)

range(0, 10)

In [2]:
for val in range(10):
    print(val)

0
1
2
3
4
5
6
7
8
9


In [4]:
lst = [1, 2]
bool(lst)

True

In [6]:
import time
from collections import deque

N = 50000

lst = list(range(N))
start = time.perf_counter()
while lst:
    lst.pop(0)                    # O(n) each — shifts the whole list left
list_time = time.perf_counter() - start

dq = deque(range(N))
start = time.perf_counter()
while dq:
    dq.popleft()                  # O(1) each
deque_time = time.perf_counter() - start

print(f"list.pop(0)    x{N}: {list_time*1000:8.1f} ms")
print(f"deque.popleft  x{N}: {deque_time*1000:8.1f} ms")
print(f"\ndeque was ~{list_time/deque_time:.0f}x faster as a queue")

list.pop(0)    x50000:   2751.3 ms
deque.popleft  x50000:      2.6 ms

deque was ~1068x faster as a queue


| | Stack (LIFO) | Queue (FIFO) |
|---|---|---|
| Add | push (top) | enqueue (back) |
| Remove | pop (top) | dequeue (front) |
| Order | newest first | oldest first |
| Python tool | `list` (`append`/`pop`) | `deque` (`append`/`popleft`) |
| Finance analogy | rollback / undo log | application processing pipeline |

## Trees
Non linear , hierarchical

In [7]:
class BSTNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None

    def insert(self, key):
        self.root = self._insert(self.root, key)

    def _insert(self, node, key):
        if node is None:
            return BSTNode(key)
        if key < node.key:
            node.left = self._insert(node.left, key)
        elif key > node.key:
            node.right = self._insert(node.right, key)
        return node                         # duplicates ignored

    def search(self, key):                  # O(height) — discard half each step
        node = self.root
        while node is not None:
            if key == node.key:
                return True
            node = node.left if key < node.key else node.right
        return False

    def inorder(self):                      # left, node, right -> SORTED order
        out = []
        def walk(n):
            if n:
                walk(n.left)
                out.append(n.key)
                walk(n.right)
        walk(self.root)
        return out

# Insert some account numbers (deliberately unsorted on the way in).
bst = BST()
for acct in [4200, 1500, 9800, 320, 75, 6000]:
    bst.insert(acct)

print("in-order (always sorted):", bst.inorder())
print("search 9800 ->", bst.search(9800))
print("search 5000 ->", bst.search(5000))

in-order (always sorted): [75, 320, 1500, 4200, 6000, 9800]
search 9800 -> True
search 5000 -> False


In [ ]:
def preorder(node, out):
    if node:
        out.append(node.key); preorder(node.left, out); preorder(node.right, out)
    return out

def postorder(node, out):
    if node:
        postorder(node.left, out); postorder(node.right, out); out.append(node.key)
    return out

print("pre-order :", preorder(bst.root, []))   # root first
print("in-order  :", bst.inorder())            # sorted
print("post-order:", postorder(bst.root, []))  # root last

pre-order : [4200, 1500, 320, 75, 9800, 6000]
in-order  : [75, 320, 1500, 4200, 6000, 9800]
post-order: [75, 320, 1500, 6000, 9800, 4200]


#### AVL Tree

In [9]:
class AVLNode:
    def __init__(self, key):
        self.key = key
        self.left = self.right = None
        self.height = 1

def _h(n):  return n.height if n else 0
def _bf(n): return _h(n.left) - _h(n.right) if n else 0

def _update(n):
    n.height = 1 + max(_h(n.left), _h(n.right))

def _rotate_right(y):           # right rotation (fixes left-heavy)
    x = y.left
    y.left = x.right
    x.right = y
    _update(y); _update(x)
    return x

def _rotate_left(x):            # left rotation (fixes right-heavy)
    y = x.right
    x.right = y.left
    y.left = x
    _update(x); _update(y)
    return y

def avl_insert(node, key):
    if node is None:
        return AVLNode(key)
    if key < node.key:
        node.left = avl_insert(node.left, key)
    else:
        node.right = avl_insert(node.right, key)
    _update(node)
    bf = _bf(node)
    if bf > 1 and key < node.left.key:        # Left-Left
        return _rotate_right(node)
    if bf < -1 and key > node.right.key:      # Right-Right
        return _rotate_left(node)
    if bf > 1 and key > node.left.key:        # Left-Right
        node.left = _rotate_left(node.left);  return _rotate_right(node)
    if bf < -1 and key < node.right.key:      # Right-Left
        node.right = _rotate_right(node.right); return _rotate_left(node)
    return node

# Insert SORTED keys 1..7 — the worst case for a plain BST.
root = None
for k in range(1, 8):
    root = avl_insert(root, k)

print("inserted 1..7 in order")
print("AVL height:", _h(root), " (a plain BST would be height 7)")
print("balanced root key:", root.key, "-> tree stayed bushy")

inserted 1..7 in order
AVL height: 3  (a plain BST would be height 7)
balanced root key: 4 -> tree stayed bushy


In [ ]:
# Counterparty transfer network. A -> B means money flowed A to B.
graph = {
    "A": ["B", "C"],
    "B": ["D"],
    "C": ["D", "E"],
    "D": ["F"],
    "E": ["F"],
    "F": [],
}

def add_edge(g, src, dst):        # an O(1) operation on the adjacency list
    g.setdefault(src, []).append(dst)
    g.setdefault(dst, [])

print("nodes:", list(graph))
print("A sends to:", graph["A"])
print("C sends to:", graph["C"])

nodes: ['A', 'B', 'C', 'D', 'E', 'F']
A sends to: ['B', 'C']
C sends to: ['D', 'E']


In [ ]:
from collections import deque

def bfs(g, start):
    visited, order = {start}, []
    q = deque([start])            # a QUEUE -> level-by-level
    while q:
        node = q.popleft()
        order.append(node)
        for nbr in g[node]:
            if nbr not in visited:
                visited.add(nbr)
                q.append(nbr)
    return order

def dfs(g, start):
    visited, order = set(), []
    stack = [start]               # a STACK -> deep-first
    while stack:
        node = stack.pop()
        if node not in visited:
            visited.add(node)
            order.append(node)
            for nbr in reversed(g[node]):   # reversed -> natural left-to-right
                if nbr not in visited:
                    stack.append(nbr)
    return order

print("BFS from A:", bfs(graph, "A"))   # level by level
print("DFS from A:", dfs(graph, "A"))   # down each path